Instalamos las librerías y dependencias necesarias

In [ ]:
!pip install fire -q
!pip install lmdb -q
!pip install opencv-python -q
!pip install natsort -q
!pip install nltk -q
!pip install numpy==1.26.4 -q
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0+cu118 --index-url https://download.pytorch.org/whl/cu118 -q
# !pip install easyocr -q
# !pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cpu -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.8/297.8 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 107.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 552.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 114.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 9.1 MB/s eta 0:00:00


Reiniciamos el entorno de la sesión y luego ejecutamos la celda siguiente para comprobar que no hay problemas entre versiones y que se instaló torch y torchvision correctamente. Si da error, es probablemente por incompatibilidades entre versiones

In [ ]:
import torch
import torchvision
import sys
import cv2
import os
#import easyocr
import albumentations as A
import csv
import pandas as pd

print(torch.__version__)
print(torchvision.__version__)

2.1.0+cu118
0.16.0+cu118


Clonamos el repositorio para entrenar el modelo de OCR

In [ ]:
!git clone https://github.com/JaidedAI/EasyOCR.git

Cloning into 'EasyOCR'...
remote: Enumerating objects: 2750, done.
remote: Counting objects: 100% (869/869), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 2750 (delta 790), reused 763 (delta 763), pack-reused 1881 (from 1)
Receiving objects: 100% (2750/2750), 157.82 MiB | 24.33 MiB/s, done.
Resolving deltas: 100% (1690/1690), done.
Updating files: 100% (313/313), done.


Debemos subir nuestro dataset con las imagenes y el archivo csv con las etiquetas y descomprimirlo dentro de `/content/EasyOCR/trainer/all_data/datasetOCR/`

In [ ]:
!unzip -q /content/datasetOCR.zip -d /content/EasyOCR/trainer/all_data/datasetOCR/

replace /content/EasyOCR/trainer/all_data/datasetOCR/labels.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


Cambiamos de directorio al modulo de entrenamiento del repositorio

In [ ]:
%cd /content/EasyOCR/trainer

/content/EasyOCR/trainer


Tambien debemos subir el siguiente archivo .yaml: `/content/EasyOCR/trainer/config_files/train_ocr_config.yaml`, el cual contiene las configuraciones para el entrenamiento del modelo.

Además, tambien debemos subir el archivo del modelo preentrenado que vamos a finetunear si es el caso, dentro de la carpeta `/content/EasyOCR/trainer/saved_models/` (esta ruta se indica dentro del .yaml tambien, deben coincidir.

### Entrenamiento

#### Conjunto de validación y Data augmentation

Primero tomaremos un subconjunto de los datos de nuestro dataset para utilizarlos en la validación del entrenamiento. Además, como tenemos muy poca cantidad de datos comparado a lo ideal para un buen entrenamiento, haremos data augmentation con distintas transformaciones.

In [ ]:
# Función para borrar y recrear una carpeta
def reset_folder(folder_path):
    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)  # Borra la carpeta y todo su contenido
    os.makedirs(folder_path)        # La recrea vacía

In [ ]:
# Leemos el csv de nuestro dataset como un dataframe. Y luego separamos entre conjunto de entrenamiento y validacion
dataset_folder = '/content/EasyOCR/trainer/all_data/datasetOCR/'
labels = os.path.join(dataset_folder, 'labels.csv')
df_original = pd.read_csv(labels)
df_validate = df_original.sample(frac=0.1, random_state=42)
df_train = df_original.drop(df_validate.index).reset_index(drop=True)
df_validate.reset_index(drop=True, inplace=True)

# Ruta donde está el dataset, y donde guardaremos el conjunto de entrenamiento y validacion
train_folder = '/content/EasyOCR/trainer/all_data/train/'
validate_folder = '/content/EasyOCR/trainer/all_data/validation/'

# Creamos las carpetas si no existen
reset_folder(train_folder)
reset_folder(validate_folder)

# Movemos todas las imagenes presentes en df_validate a la carpeta validate_folder
for index, row in df_validate.iterrows():
    img_path = os.path.join(dataset_folder, row['filename'])
    os.rename(img_path, os.path.join(validate_folder, row['filename']))

# Y analogamente para las imagenes presentes en df_train
for index, row in df_train.iterrows():
    img_path = os.path.join(dataset_folder, row['filename'])
    os.rename(img_path, os.path.join(train_folder, row['filename']))

# Guardamos el dataframe de validacion como csv
df_validate.to_csv(os.path.join(validate_folder, 'labels.csv'), index=False)

### Data augmentation ###

# Definimos el pipeline de augmentacion
transform = A.Compose([
    A.Rotate(limit=4, border_mode=cv2.BORDER_REPLICATE, p=0.3),  # máximo 5 grados
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.4, p=0.5),  # brillo/contraste
    A.MotionBlur(blur_limit=21, p=0.4),  # desenfoque máximo de 7 píxeles
    A.Affine(translate_percent=0.02, scale=(0.7, 1.3), rotate=(-4, 4), p=0.5)  # leve shift/zoom/rotación
])

# iteramos por cada fila en df_train
for index, row in df_train.iterrows():
    # Path donde se deben guardar las imagenes augmentadas de la imagen en esta fila
    img_path = os.path.join(train_folder, row['filename'])
    image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        print(f"Advertencia: No se pudo cargar la imagen {img_path}")
        continue

    # generar 5 imagenes aumentadas por imagen de entrenamiento
    for i in range(40):
        # Aplicamos la transformacion y creamos nuevo nombre a esta imagen
        augmented = transform(image=image)['image']
        augmented_filename = f"{os.path.splitext(row['filename'])[0]}_aug_{i}.png"
        # Guardamos la imagen en la ruta correspondiente (dentro del dataset de entrenamiento)
        cv2.imwrite(os.path.join(train_folder, augmented_filename), augmented)
        # Nueva fila a agregar en el dataframe de entrenamiento
        new_row = {'filename': augmented_filename, 'words': row['words']}
        # Asignar un nuevo índice manualmente y guardamos la nueva fila
        df_train.loc[len(df_train)] = new_row

# guardamos el archivo labels que contiene las etiquetas de las imagenes de entrenamiento
df_train.to_csv(os.path.join(train_folder, 'labels.csv'), index=False)

In [ ]:
!rm -rf /content/EasyOCR/trainer/all_data/datasetOCR/

In [ ]:
!ls /content/EasyOCR/trainer/all_data/validation/ipynb_checkpoints

ls: cannot access '/content/EasyOCR/trainer/all_data/validation/ipynb_checkpoints': No such file or directory


Antes de entrenar es necesario modificar un poco el codigo:

* En `dataset.py`:
    - linea 164: if re.search(out_of_char, label)
    - linea 101 y 106: next(...) en lugar de ....next()

Añadimos esa ruta al path del sistema para que lo detecte como un modulo y poder importar funciones dentro de archivos en esa carpeta

Ojito con "freeze_FeatureFxtraction" y "freeze_FeatureExtraction". En el codigo está como lo primero. En el Yaml debe estar igual, o sino corregir el codigo.

In [ ]:
sys.path.insert(0, '/content/EasyOCR/trainer')

In [ ]:
import torch.backends.cudnn as cudnn
import yaml
from train import train
from utils import AttrDict

cudnn.benchmark = True
cudnn.deterministic = False

In [ ]:
def get_config(file_path):
    with open(file_path, 'r', encoding="utf8") as stream:
        opt = yaml.safe_load(stream)
    opt = AttrDict(opt)
    if opt.lang_char == 'None':
        characters = ''
        for data in opt['select_data'].split('-'):
            csv_path = os.path.join(opt['train_data'], data, 'labels.csv')
            df = pd.read_csv(csv_path, sep='^([^,]+),', engine='python', usecols=['filename', 'words'], keep_default_na=False)
            all_char = ''.join(df['words'])
            characters += ''.join(set(all_char))
        characters = sorted(set(characters))
        opt.character= ''.join(characters)
    else:
        opt.character = opt.number + opt.symbol + opt.lang_char
    os.makedirs(f'./saved_models/{opt.experiment_name}', exist_ok=True)
    return opt

In [ ]:
opt = get_config("config_files/train_ocr_config.yaml")

In [ ]:
opt

{'number': '0123456789',
 'symbol': '',
 'lang_char': 'abcdefghijklmnopqrstuvwxyz',
 'experiment_name': 'plates_ocr_50_batch12_c',
 'train_data': 'all_data/train',
 'valid_data': 'all_data/validation',
 'manualSeed': 1111,
 'workers': 2,
 'batch_size': 12,
 'num_iter': 10000,
 'valInterval': 100,
 'saved_model': 'saved_models/None-VGG-BiLSTM-CTC.pth',
 'FT': True,
 'optim': False,
 'lr': 1.0,
 'beta1': 0.9,
 'rho': 0.95,
 'eps': 1e-08,
 'grad_clip': 5,
 'select_data': '',
 'batch_ratio': '1',
 'total_data_usage_ratio': 1.0,
 'batch_max_length': 6,
 'imgH': 64,
 'imgW': 200,
 'rgb': False,
 'sensitive': False,
 'PAD': True,
 'contrast_adjust': 0.0,
 'data_filtering_off': False,
 'Transformation': 'None',
 'FeatureExtraction': 'VGG',
 'SequenceModeling': 'BiLSTM',
 'Prediction': 'CTC',
 'num_fiducial': 20,
 'input_channel': 1,
 'output_channel': 512,
 'hidden_size': 256,
 'decode': 'greedy',
 'new_prediction': False,
 'freeze_FeatureFxtraction': False,
 'freeze_SequenceModeling': False,


In [ ]:
print(len('0123456789ABCDFGHJKLMNOPRSTUVWXYZ'))
print(len('0123456789abcdefghijklmnopqrstuvwxyz'))

33
36


In [ ]:
# amp es Automatic Mixed Precision, que reduce floats de 32 bits de floats de 16 bits
# cuando es posible para ahorrar memoria (no es necesario en este caso)
train(opt, amp=False)

Filtering the images containing characters which are not in opt.character
Filtering the images whose label is longer than opt.batch_max_length
--------------------------------------------------------------------------------
dataset_root: all_data/train
opt.select_data: ['']
opt.batch_ratio: ['1']
--------------------------------------------------------------------------------
dataset_root:    all_data/train	 dataset: 
all_data/train/
sub-directory:	/.	 num samples: 14965
num total samples of : 14965 x 1.0 (total_data_usage_ratio) = 14965
num samples of  per batch: 12 x 1.0 (batch_ratio) = 12
--------------------------------------------------------------------------------
Total_batch_size: 12 = 12
--------------------------------------------------------------------------------
dataset_root:    all_data/validation	 dataset: /
all_data/validation/
sub-directory:	/.	 num samples: 41
--------------------------------------------------------------------------------
No Transformation module sp

SystemExit: 

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Descargar la carpeta con el modelo entrenado
import shutil
from google.colab import files

# Comprimir la carpeta
shutil.make_archive('plates_ocr_50_batch12_a', 'zip', '/content/EasyOCR/trainer/saved_models/plates_ocr_50_batch12_a')

# Descargar el archivo ZIP
files.download('plates_ocr_50_batch12_a.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Cargar modelo y probar el OCR

Creamos los directorios donde debemos colocar los archivos necesarios:
* custom_model.py (~/.EasyOCR/user_network)
* custom_model.yaml (~/.EasyOCR/user_network)
* custom_model.pth (~/.EasyOCR/model)

In [ ]:
!mkdir -p ~/.EasyOCR/user_network
!mkdir -p ~/.EasyOCR/model

Ahora copiamos esos 3 archivos que subimos a las carpetas creadas y que se indicaron arriba

In [ ]:
!cp /content/custom_model_samepred.py /root/.EasyOCR/user_network/custom_model.py
!cp /content/custom_model_samepred.yaml /root/.EasyOCR/user_network/custom_model.yaml
!cp /content/custom_model_samepred.pth /root/.EasyOCR/model/custom_model.pth

In [ ]:
# modulos presentes en el repo deep-text-recognition-benchmark necesarios para custom_model.py
!cp -r /content/EasyOCR/trainer/modules /root/.EasyOCR/user_network/

In [ ]:
!ls /root/.EasyOCR/user_network/

custom_model.py  custom_model.yaml  modules


In [ ]:
# Borrar las carpetas creadas anteriormente para copiar los datos de nuevo

!rm -r /root/.EasyOCR/user_network/
!rm -r /root/.EasyOCR/model/

Finalmente cargamos el modelo y testeamos

In [ ]:
original_reader = easyocr.Reader(lang_list=['en'])

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
custom_reader = easyocr.Reader(lang_list=['en'], recog_network='custom_model')

Progress: |██████████████████████████████████████████████████| 100.0% CompleteNo Transformation module specified


In [ ]:
# Prueba con una imagen
image_path = '/content/ocr_3_20250613_105056.png'
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
results = original_reader.readtext(image, detail=1, adjust_contrast=0.0)
print(f"Resultados original: {results}")
results = custom_reader.recognize(image, detail=1, adjust_contrast=0.0)
print(f"Resultados custom: {results}")


Resultados original: [([[0, 9], [293, 9], [293, 117], [0, 117]], 'RVHL 981', 0.30133506546998134)]
Resultados custom: [([[0, 0], [340, 0], [340, 140], [0, 140]], 'rvhl98', 0.9205440912122198)]
